<a href="https://colab.research.google.com/github/ThisalFernando/Fake-Real-News-Detection/blob/Multinominal_Naive_Bayes/Multinominal_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Imports + Mount Drive + Load dataset (Google Colab)

In [2]:
import pandas as pd
import numpy as np

from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, classification_report, confusion_matrix
)

import matplotlib.pyplot as plt

# 1) Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2) Set the dataset path
CSV_PATH = "/content/drive/MyDrive/improved_fake_news_dataset2.csv"

# 3) Load dataset
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(5)

Mounted at /content/drive
Shape: (123954, 4)
Columns: ['id', 'title', 'text', 'label']


,id,title,text,label
0,156735,NaN,Embargo on Phillipines - withdraw our military...,1
1,212890,Especially sea lay son hear.,too win central main bag save common she famil...,1
2,183407,Russian oil giant Rosneft to restore Romanov p...,Russian oil giant Rosneft to restore Romanov p...,1
3,199095,NaN,the club world cup is here our representative ...,0
4,228061,NaN,"the ismaili club s board of directors, headed ...",1


# Cell 2: Basic cleaning + combine title and text

In [3]:
TITLE_COL = "title"
TEXT_COL = "text"
LABEL_COL = "label"

# Check required columns
missing_cols = [c for c in [TITLE_COL, TEXT_COL, LABEL_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}. Found columns: {df.columns.tolist()}")

# Fill NaNs and combine text
df[TITLE_COL] = df[TITLE_COL].fillna("")
df[TEXT_COL] = df[TEXT_COL].fillna("")
df["content"] = (df[TITLE_COL].astype(str) + " " + df[TEXT_COL].astype(str)).str.strip()

# Ensure label is numeric 0/1
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df.dropna(subset=[LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

print("After cleaning shape:", df.shape)
print("Label value counts:\n", df[LABEL_COL].value_counts())
df[["content", LABEL_COL]].head(3)

After cleaning shape: (123954, 5)
Label value counts:
 label
1    61977
0    61977
Name: count, dtype: int64


,content,label
0,Embargo on Phillipines - withdraw our military...,1
1,Especially sea lay son hear. too win central m...,1
2,Russian oil giant Rosneft to restore Romanov p...,1


# Cell 3: Train/Test split

In [4]:
X = df["content"].values
y = df[LABEL_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

print("Train label distribution:", np.bincount(y_train))
print("Test label distribution:", np.bincount(y_test))

Train size: 99163
Test size: 24791
Train label distribution: [49581 49582]
Test label distribution: [12396 12395]


# Cell 4: Build model pipeline and train

In [5]:
model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ("nb", MultinomialNB(alpha=1.0))
])

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.
